In [ ]:
# Load R magic extension for Python Jupyter kernel (Kaggle / Colab support)
try:
    %load_ext rpy2.ipython
except Exception as e:
    print("Note on rpy2 initialization:", e)

# 4-Class ESI (2, 3, 4, 5) LightGBM Classifier with Per-Class Configurable KDE Upsampling (`models/lightbgm_feng_esi2345_extreme.ipynb`)

This notebook trains a **4-Class LightGBM Multi-Class Classifier** for **ESI 2, ESI 3, ESI 4, and ESI 5** (excluding ESI 1 rows) using **13 Predictor Features** (`age`, `gender`, `cc_breathingdifficulty` + 10 Clinical Feature Engineered Vital Flags) with **Per-Class Multivariate Kernel Density Estimation (KDE) Synthetic Sample Generation**, fully configurable with separate target percentages for each class relative to the largest class:

### System Architecture & Workflow
1. **Filtering & 4-Class Target Definition**: Removes ESI 1 rows completely, defining 4 target classes (`"2"`, `"3"`, `"4"`, `"5"`).
2. **Predictor Feature Selection (13 Predictors)**:
   - **Raw Predictors (From `triage_conf.json`)**: `age`, `gender`, `cc_breathingdifficulty`.
   - **10 Clinical Feature Engineered Flags**: `is_dyspnea_total`, `is_dyspnea_moderate`, `is_bradypnea`, `is_tachypnea`, `is_hypotension`, `is_hypertension`, `is_bradycardia_total`, `is_bradycardia_moderate`, `is_tachycardia_total`, `is_tachycardia_moderate`.
3. **Stratified Data Partitioning First**: Splits dataset into Train (70%), Validation (15%), and Test (15%) splits prior to scaling to prevent data leakage.
4. **Separate Per-Class KDE Upsampling Percentages**: Target sample count for each separate class is individually configured via `kde_class_pcts` as a percentage of the largest class count (e.g. `c("2" = 1.0, "3" = 1.0, "4" = 1.0, "5" = 1.0)` for 1:1:1:1 equal balance).
5. **Multi-Class LightGBM Gradient Boosting**: Fits 4-class decision trees (`objective = "multiclass"`, `num_class = 4`, `metric = "multi_logloss"`) via `lgb.Dataset` and `lgb.train()`.
6. **Comprehensive 4x4 Benchmarking Across Splits**: Evaluates Train, Validation, and Test performance with 4x4 confusion matrices, Class Count Comparison Tables, Accuracy, Macro/Per-Class Precision, Recall (Sensitivity), F1 Score, PR-AUC, and ROC-AUC.
7. **Reports & Artifacts**:
   - **Diagnostic Plots**: Metrics bar chart (`plots/lightbgm_feng_esi2345_metrics_barchart.png`).
   - **CSV Reports**: `reports/lightbgm_feng_esi2345_val_report.csv`, `reports/lightbgm_feng_esi2345_test_report.csv`.
   - **Model Export**: Saved to `deploy/lightbgm_feng_esi2345_extreme_model.rds`.

In [ ]:
%%R
# ---------------------------------------------------------
# Step 1: Load Required Libraries & Parse Configuration JSON
# ---------------------------------------------------------
suppressPackageStartupMessages({
  library(jsonlite)
  library(caret)
  library(dplyr)
  library(ggplot2)
  library(tidyr)
  library(pROC)
  library(lightgbm)
})
config_path <- "../config/triage_conf.json"
if (!file.exists(config_path)) {
  config_path <- "config/triage_conf.json"
}
config <- fromJSON(config_path)
cat("=== Configuration Loaded from config/triage_conf.json ===\n")
cat("Data Source Path:", config$path$data_source, "\n")
cat("Target Column:   ", config$classes$target_col, "\n")
cat("Test Size:       ", config$training$test_size, "\n")
cat("Val Size:        ", config$training$val_size, "\n")
cat("Random State:    ", config$training$random_state, "\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 2: Load Data, Exclude ESI 1 Rows & Construct 13 Predictor Features
# ---------------------------------------------------------
set.seed(config$training$random_state)
data_file <- config$path$data_source
if (!file.exists(data_file) && file.exists(paste0("../", data_file))) {
  data_file <- paste0("../", data_file)
}
cat("Loading dataset from:", data_file, "...\n")
data_env <- new.env()
load(data_file, envir = data_env)
df_names <- ls(data_env)[sapply(ls(data_env), function(x) is.data.frame(get(x, envir = data_env)))]
df_sizes <- sapply(df_names, function(x) nrow(get(x, envir = data_env)))
data_obj_name <- df_names[which.max(df_sizes)]
cat(sprintf("Selected main dataset object: '%s' (%d rows)\n", data_obj_name, max(df_sizes)))
raw_df <- get(data_obj_name, envir = data_env)
target_col <- config$classes$target_col
# FILTERING: Remove ESI 1 rows completely
raw_esi_all <- as.character(raw_df[[target_col]])
keep_mask   <- raw_esi_all != "1"
raw_df  <- raw_df[keep_mask, ]
raw_esi <- raw_esi_all[keep_mask]
cat(sprintf("ESI 1 Filtering: Removed %d ESI 1 rows (Remaining rows: %d)\n", sum(!keep_mask), nrow(raw_df)))
gender_vec <- if ("gender" %in% names(raw_df)) ifelse(as.character(raw_df$gender) == "Male", 1, 0) else 0
cc_bd_vec  <- if ("cc_breathingdifficulty" %in% names(raw_df)) ifelse(!is.na(raw_df$cc_breathingdifficulty), raw_df$cc_breathingdifficulty, 0) else 0
# Construct 13 Predictors: age, gender, cc_breathingdifficulty + 10 Clinical FE Vital Anomaly Flags
df_full <- data.frame(
  age                     = raw_df$age,
  gender                  = gender_vec,
  cc_breathingdifficulty  = cc_bd_vec,
  is_dyspnea_total        = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 < 90, 1, 0),
  is_dyspnea_moderate     = ifelse(!is.na(raw_df$triage_vital_o2) & raw_df$triage_vital_o2 >= 90 & raw_df$triage_vital_o2 < 94, 1, 0),
  is_bradypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr < 10, 1, 0),
  is_tachypnea            = ifelse(!is.na(raw_df$triage_vital_rr) & raw_df$triage_vital_rr > 30, 1, 0),
  is_hypotension          = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp <= 90, 1, 0),
  is_hypertension         = ifelse(!is.na(raw_df$triage_vital_sbp) & raw_df$triage_vital_sbp > 220, 1, 0),
  is_bradycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr < 40, 1, 0),
  is_bradycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr >= 40 & raw_df$triage_vital_hr < 60, 1, 0),
  is_tachycardia_total    = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 150, 1, 0),
  is_tachycardia_moderate = ifelse(!is.na(raw_df$triage_vital_hr) & raw_df$triage_vital_hr > 100 & raw_df$triage_vital_hr <= 150, 1, 0)
)
# Target Definition: 4 Classes ('2', '3', '4', '5')
df_full$target_layer1 <- factor(raw_esi, levels = c("2", "3", "4", "5"))
initial_rows <- nrow(df_full)
df_full <- na.omit(df_full)
cat(sprintf("Complete Case Filtering: Removed %d rows with NULL/NA features (Remaining complete rows: %d)\n",
            initial_rows - nrow(df_full), nrow(df_full)))
cat(sprintf("Full Feature Dataset Ready (Pre-Partitioning): %d total rows x %d cols\n", nrow(df_full), ncol(df_full)))
cat("Predictor Features Included (13 Total):\n")
print(setdiff(names(df_full), "target_layer1"))
cat("\nNatural 4-Class Target Distribution ('2', '3', '4', '5'):\n")
print(table(df_full$target_layer1))

In [ ]:
%%R
# ---------------------------------------------------------
# Step 3: Stratified Partitioning & Separate Per-Class KDE Upsampling
# ---------------------------------------------------------
set.seed(config$training$random_state)
# Pure Base R Multivariate Normal Generator
rmvnorm_base <- function(n, Sigma) {
  d <- ncol(Sigma)
  eig <- eigen(Sigma, symmetric = TRUE)
  evals <- pmax(eig$values, 1e-8)
  transform <- eig$vectors %*% diag(sqrt(evals), d)
  Z <- matrix(rnorm(n * d), nrow = n, ncol = d)
  res <- Z %*% t(transform)
  colnames(res) <- colnames(Sigma)
  return(res)
}
# MULTI-CLASS KDE SYNTHETIC SAMPLE GENERATOR
generate_kde_class_samples <- function(df_cls, target_cls, n_synth_target, bandwidth_factor = 0.5) {
  if (n_synth_target <= 0 || nrow(df_cls) == 0) return(df_cls[0, , drop = FALSE])
  
  cont_cols <- intersect(c("age"), names(df_cls))
  if (length(cont_cols) == 0) return(df_cls[0, , drop = FALSE])
  
  cont_mat <- as.matrix(df_cls[, cont_cols, drop = FALSE])
  colnames(cont_mat) <- cont_cols
  n_orig   <- nrow(cont_mat)
  d        <- ncol(cont_mat)
  
  cov_mat <- cov(cont_mat)
  colnames(cov_mat) <- cont_cols
  rownames(cov_mat) <- cont_cols
  
  h_silverman <- (4 / (d + 2))^(1 / (d + 4)) * (n_orig^(-1 / (d + 4))) * bandwidth_factor
  
  sampled_indices <- sample(1:n_orig, size = n_synth_target, replace = TRUE)
  synth_df <- df_cls[sampled_indices, , drop = FALSE]
  
  noise_mat <- rmvnorm_base(n = n_synth_target, Sigma = h_silverman^2 * cov_mat)
  colnames(noise_mat) <- cont_cols
  
  synth_cont <- as.matrix(synth_df[, cont_cols, drop = FALSE]) + noise_mat
  colnames(synth_cont) <- cont_cols
  
  # Clamp age to realistic bounds [18, 110]
  if ("age" %in% cont_cols) synth_cont[, "age"] <- pmin(110, pmax(18, synth_cont[, "age"]))
  
  synth_df[, cont_cols] <- synth_cont
  synth_df$target_layer1 <- factor(target_cls, levels = levels(df_cls$target_layer1))
  return(synth_df)
}
# CONFIGURABLE SEPARATE PER-CLASS KDE UPSAMPLING PERCENTAGES
# Target sample size for each separate class as a fraction of the largest class count in training set.
# Adjust each class percentage individually below:
kde_class_pcts <- c(
  "2" = 1.0,   # Target count for ESI 2 = 100% of largest class count
  "3" = 1.0,   # Target count for ESI 3 = 100% of largest class count
  "4" = 1.0,   # Target count for ESI 4 = 100% of largest class count
  "5" = 2.5    # Target count for ESI 5 = 100% of largest class count
)
test_size <- config$training$test_size
val_size  <- config$training$val_size
# Stratified Test split (15%)
in_train_val <- createDataPartition(df_full$target_layer1, p = 1 - test_size, list = FALSE)
train_val_df <- df_full[in_train_val, ]
test_df      <- df_full[-in_train_val, ]
# Stratified Validation split (15%)
rel_val_size <- val_size / (1 - test_size)
in_train    <- createDataPartition(train_val_df$target_layer1, p = 1 - rel_val_size, list = FALSE)
train_df    <- train_val_df[in_train, ]
val_df      <- train_val_df[-in_train, ]
# APPLY SEPARATE PER-CLASS KDE UPSAMPLING TO TRAINING SET ONLY
orig_counts <- table(train_df$target_layer1)
max_class_count <- max(orig_counts)
largest_class   <- names(orig_counts)[which.max(orig_counts)]
cat(sprintf("=== Separate Per-Class KDE Upsampling Setup ===\n"))
cat(sprintf("Largest Class in Training Set: '%s' (%d rows)\n\n", largest_class, max_class_count))
kde_synth_list <- list()
for (cls in levels(train_df$target_layer1)) {
  df_cls <- train_df[train_df$target_layer1 == cls, ]
  n_orig <- nrow(df_cls)
  
  # Fetch separate percentage for this class
  pct <- if (cls %in% names(kde_class_pcts)) kde_class_pcts[[cls]] else 1.0
  
  target_total <- as.integer(max_class_count * pct)
  n_needed     <- max(0, target_total - n_orig)
  
  if (n_needed > 0) {
    cat(sprintf("  - Class '%s': Original = %d, Target = %d (%.1f%% of max) -> Generating %d KDE synthetic samples\n",
                cls, n_orig, target_total, pct * 100, n_needed))
    synth_cls <- generate_kde_class_samples(df_cls, target_cls = cls, n_synth_target = n_needed, bandwidth_factor = 0.5)
    kde_synth_list[[cls]] <- synth_cls
  } else {
    cat(sprintf("  - Class '%s': Original = %d >= Target = %d (%.1f%% of max) -> No upsampling needed\n",
                cls, n_orig, target_total, pct * 100))
  }
}
# Combine original train_df + all KDE synthetic samples
if (length(kde_synth_list) > 0) {
  all_kde_synth <- do.call(rbind, kde_synth_list)
  train_df_aug  <- rbind(train_df, all_kde_synth)
  train_df      <- train_df_aug[sample(nrow(train_df_aug)), ]
}
# Standardize continuous feature (age) AFTER KDE upsampling
cont_cols <- intersect(c("age"), names(train_df))
preproc <- preProcess(train_df[, cont_cols, drop = FALSE], method = c("center", "scale"))
train_df <- predict(preproc, train_df)
val_df   <- predict(preproc, val_df)
test_df  <- predict(preproc, test_df)
cat("\n=== Target Distributions Post-KDE Upsampling ===\n")
cat("Augmented Training Set Target Distribution:\n")
print(table(train_df$target_layer1))
cat("\nNatural Validation Set Target Distribution:\n")
print(table(val_df$target_layer1))
cat("\nNatural Test Set Target Distribution:\n")
print(table(test_df$target_layer1))
feat_names <- setdiff(names(train_df), "target_layer1")
X_train <- as.matrix(train_df[, feat_names])
y_train_fac <- train_df$target_layer1
y_train_0idx <- as.integer(y_train_fac) - 1
X_val   <- as.matrix(val_df[, feat_names])
y_val_fac   <- val_df$target_layer1
y_val_0idx   <- as.integer(y_val_fac) - 1
X_test  <- as.matrix(test_df[, feat_names])
y_test_fac  <- test_df$target_layer1
y_test_0idx  <- as.integer(y_test_fac) - 1
dtrain_lgb <- lgb.Dataset(data = X_train, label = y_train_0idx)
dval_lgb   <- lgb.Dataset.create.valid(dtrain_lgb, data = X_val, label = y_val_0idx)
dtest_lgb  <- lgb.Dataset.create.valid(dtrain_lgb, data = X_test, label = y_test_0idx)

In [ ]:
%%R
# ---------------------------------------------------------
# Step 4: Train 4-Class LightGBM Multi-Class Model
# ---------------------------------------------------------
set.seed(config$training$random_state)
cat("Training 4-Class LightGBM Model ('2', '3', '4', '5')...\n")
lgb_params <- list(
  objective        = "multiclass",
  num_class        = 4,
  metric           = "multi_logloss",
  learning_rate    = 0.05,
  num_leaves       = 31,
  feature_fraction = 0.8,
  bagging_fraction = 0.8,
  bagging_freq     = 1,
  verbosity        = -1
)
lgb_model <- lgb.train(
  params                = lgb_params,
  data                  = dtrain_lgb,
  nrounds               = 150,
  valids                = list(train = dtrain_lgb, val = dval_lgb),
  early_stopping_rounds = 20,
  verbose               = -1
)
cat("4-Class LightGBM Model Training Complete!\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 5: Comprehensive 4-Class Benchmark Across Splits & Export CSV Reports
# ---------------------------------------------------------
calc_pr_auc <- function(actual_binary, prob_positive) {
  tryCatch({
    ord <- order(prob_positive, decreasing = TRUE)
    act_sorted <- (actual_binary[ord] == 1)
    tp <- cumsum(act_sorted)
    fp <- cumsum(!act_sorted)
    n_pos <- sum(act_sorted)
    if (n_pos == 0) return(NA)
    rec <- c(0, tp / n_pos)
    prec <- c(tp[1] / max(1, tp[1] + fp[1]), tp / (tp + fp))
    dx <- diff(rec)
    my <- (prec[-1] + prec[-length(prec)]) / 2
    return(as.numeric(sum(dx * my)))
  }, error = function(e) NA)
}
evaluate_lightgbm_4class <- function(model, X_mat, actual_fac, set_name) {
  raw_preds <- predict(model, X_mat)
  
  if (is.matrix(raw_preds)) {
    prob_mat <- raw_preds
  } else {
    prob_mat <- matrix(raw_preds, ncol = 4, byrow = TRUE)
  }
  colnames(prob_mat) <- c("2", "3", "4", "5")
  
  pred_indices <- apply(prob_mat, 1, which.max)
  pred_val     <- colnames(prob_mat)[pred_indices]
  pred_fac     <- factor(pred_val, levels = c("2", "3", "4", "5"))
  act_fac      <- factor(actual_fac, levels = c("2", "3", "4", "5"))
  
  cm  <- confusionMatrix(pred_fac, act_fac)
  acc <- as.numeric(cm$overall["Accuracy"])
  
  prec_by_class <- as.numeric(cm$byClass[, "Pos Pred Value"])
  rec_by_class  <- as.numeric(cm$byClass[, "Sensitivity"])
  prec_by_class[is.na(prec_by_class)] <- 0
  rec_by_class[is.na(rec_by_class)]   <- 0
  
  pr_auc_by_class <- sapply(1:4, function(i) {
    cls_name <- levels(act_fac)[i]
    act_bin  <- ifelse(act_fac == cls_name, 1, 0)
    calc_pr_auc(act_bin, prob_mat[, i])
  })
  
  actual_counts <- as.numeric(table(act_fac))
  pred_counts   <- as.numeric(table(pred_fac))
  diff_vec      <- pred_counts - actual_counts
  diff_str      <- ifelse(diff_vec >= 0, paste0("+", diff_vec), as.character(diff_vec))
  
  report_df <- data.frame(
    Class        = levels(act_fac),
    Actual_Count = actual_counts,
    Pred_Count   = pred_counts,
    Diff         = diff_str,
    Precision    = round(prec_by_class, 4),
    Recall       = round(rec_by_class, 4),
    PR_AUC       = round(pr_auc_by_class, 4)
  )
  
  macro_prec   <- mean(prec_by_class)
  macro_rec    <- mean(rec_by_class)
  macro_pr_auc <- mean(pr_auc_by_class, na.rm = TRUE)
  
  cat(sprintf("============================================================\n"))
  cat(sprintf("   4-CLASS LIGHTGBM (2, 3, 4, 5) - %s SET BENCHMARK\n", toupper(set_name)))
  cat(sprintf("============================================================\n"))
  cat(sprintf("  Overall Accuracy     : %.4f (%.2f%%)\n", acc, acc * 100))
  cat(sprintf("  Macro Precision      : %.4f\n", macro_prec))
  cat(sprintf("  Macro Recall (Sens)  : %.4f\n", macro_rec))
  cat(sprintf("  Macro PR-AUC         : %.4f\n", macro_pr_auc))
  cat(sprintf("============================================================\n\n"))
  
  cat("Per-Class Count Comparison & Performance Summary:\n")
  print(report_df)
  cat("\nConfusion Matrix (Rows: Predicted, Columns: Actual):\n")
  print(cm$table)
  cat(sprintf("============================================================\n\n"))
  
  return(list(acc = acc, macro_prec = macro_prec, macro_rec = macro_rec, macro_pr_auc = macro_pr_auc, prob_mat = prob_mat, report_df = report_df))
}
res_train <- evaluate_lightgbm_4class(lgb_model, X_train, train_df$target_layer1, "Train")
res_val   <- evaluate_lightgbm_4class(lgb_model, X_val,   val_df$target_layer1,   "Validation")
res_test  <- evaluate_lightgbm_4class(lgb_model, X_test,  test_df$target_layer1,  "Test")
# Write CSV Reports
reports_dir <- "../reports"
if (!dir.exists(reports_dir)) reports_dir <- "reports"
if (!dir.exists(reports_dir)) dir.create(reports_dir, recursive = TRUE)
write.csv(res_val$report_df,  file = file.path(reports_dir, "lightbgm_feng_esi2345_val_report.csv"),  row.names = FALSE)
write.csv(res_test$report_df, file = file.path(reports_dir, "lightbgm_feng_esi2345_test_report.csv"), row.names = FALSE)
cat("Validation CSV Report written to: reports/lightbgm_feng_esi2345_val_report.csv\n")
cat("Test CSV Report written to:       reports/lightbgm_feng_esi2345_test_report.csv\n")

In [ ]:
%%R
# ---------------------------------------------------------
# Step 6: Diagnostic Plots (Metrics Bar Chart)
# ---------------------------------------------------------
plots_dir <- "../plots"
if (!dir.exists(plots_dir)) plots_dir <- "plots"
if (!dir.exists(plots_dir)) dir.create(plots_dir, recursive = TRUE)
metrics_summary <- data.frame(
  Split           = factor(c("Train", "Validation", "Test"), levels = c("Train", "Validation", "Test")),
  Accuracy        = c(res_train$acc,        res_val$acc,        res_test$acc),
  Macro_Precision = c(res_train$macro_prec,  res_val$macro_prec,  res_test$macro_prec),
  Macro_Recall    = c(res_train$macro_rec,   res_val$macro_rec,   res_test$macro_rec),
  Macro_PR_AUC    = c(res_train$macro_pr_auc,res_val$macro_pr_auc,res_test$macro_pr_auc)
)
metrics_long <- metrics_summary %>%
  pivot_longer(cols = c("Accuracy", "Macro_Precision", "Macro_Recall", "Macro_PR_AUC"), names_to = "Metric", values_to = "Score")
p_bar <- ggplot(metrics_long, aes(x = Metric, y = Score, fill = Split)) +
  geom_bar(stat = "identity", position = position_dodge(width = 0.7), width = 0.6) +
  geom_text(aes(label = sprintf("%.3f", Score)), position = position_dodge(width = 0.7), vjust = -0.3, size = 3) +
  theme_minimal() +
  scale_fill_manual(values = c("Train" = "#2b5c8f", "Validation" = "#e07a5f", "Test" = "#81b29a")) +
  labs(title = "Train vs. Validation vs. Test Metrics (4-Class LightGBM ESI 2,3,4,5)",
       subtitle = "Comparing Overall Accuracy, Macro Precision, Macro Recall, and Macro PR-AUC",
       y = "Metric Value Score", x = "") +
  theme(plot.title = element_text(face = "bold", size = 13), legend.position = "top")
ggsave(file.path(plots_dir, "lightbgm_feng_esi2345_metrics_barchart.png"), plot = p_bar, width = 9.5, height = 5, dpi = 300)
cat("Metrics Comparison Bar Chart saved to: plots/lightbgm_feng_esi2345_metrics_barchart.png\n")
p_bar

In [ ]:
%%R
# ---------------------------------------------------------
# Step 7: Save 4-Class LightGBM Model Artifact
# ---------------------------------------------------------
deploy_dir <- "../deploy"
if (!dir.exists(deploy_dir)) dir.create(deploy_dir, recursive = TRUE)
model_path <- file.path(deploy_dir, "lightbgm_feng_esi2345_extreme_model.rds")
saveRDS(list(model = lgb_model, preproc = preproc, class_levels = c("2", "3", "4", "5"), kde_class_pcts = kde_class_pcts), file = model_path)
cat("4-Class LightGBM ESI 2,3,4,5 model saved to:", model_path, "\n")